# Manual Graph RAG Pipeline with Neptune Analytics

This notebook demonstrates how to build a custom Graph RAG pipeline using Neptune Analytics directly,
bypassing Bedrock Knowledge Bases managed ingestion. This gives you full control over:

- **Which model** extracts entities and relationships (any model available via Bedrock)
- **Graph schema** design (node/edge types, properties)
- **Chunking strategy**
- **Query patterns** (hybrid vector + graph traversal)

### Pipeline Steps
1. Setup and document loading
2. Chunk documents
3. Extract entities and relationships using Claude Haiku 4.5
4. Create Neptune Analytics graph
5. Insert nodes, edges, and embeddings
6. Query: vector similarity + graph traversal
7. Generate answers with retrieved context
8. Cleanup

## 1. Setup

In [ ]:
%pip install -r ../requirements.txt --quiet

In [ ]:
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [1]:
import boto3
import json
import time
import concurrent.futures
from collections import Counter
from botocore.client import Config
from pypdf import PdfReader
from IPython.display import display, Markdown

# Clients
sts_client = boto3.client('sts')
session = boto3.session.Session()
region = session.region_name
account_id = sts_client.get_caller_identity()['Account']

bedrock_runtime = boto3.client('bedrock-runtime', region_name=region)
neptune_client = boto3.client('neptune-graph', region_name=region)

# Models
EXTRACTION_MODEL = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
GENERATION_MODEL = "us.anthropic.claude-sonnet-4-6"
EMBEDDING_DIMENSION = 1024

print(f"Region: {region}")
print(f"Account: {account_id}")
print(f"Extraction model: {EXTRACTION_MODEL}")
print(f"Generation model: {GENERATION_MODEL}")

Region: us-east-1
Account: 678717774493
Extraction model: us.anthropic.claude-haiku-4-5-20251001-v1:0
Generation model: us.anthropic.claude-sonnet-4-6


## 2. Load and Chunk Documents

In [24]:
# Load the synthetic financial report
pdf_path = "../synthetic_dataset/octank_financial_10K.pdf"
reader = PdfReader(pdf_path)

full_text = ""
for page in reader.pages:
    full_text += page.extract_text() + "\n"

print(f"Loaded {len(reader.pages)} pages, {len(full_text)} characters")

Loaded 120 pages, 226936 characters


In [25]:
def chunk_text(text, chunk_size=1500, overlap=200):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append({
                "id": f"chunk_{len(chunks)}",
                "text": chunk.strip(),
                "start": start,
                "end": end
            })
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(full_text)
print(f"Created {len(chunks)} chunks")
print(f"Sample chunk: {chunks[0]['text'][:200]}...")

Created 175 chunks
Sample chunk: EXHIBITS, FINANCIAL STATEMENT SCHEDULES including Financial
Statement Schedules, Exhibits, Signatures, Power of Attorney
Octank Financial's 10K report includes several financial statement schedules an...


## 3. Extract Entities and Relationships (Two-Pass Pipeline)

**Why two passes?** A single-pass extraction lets the LLM invent new entity names in every chunk,
producing duplicates like "Credit Risk" / "credit risk" / "Credit risk". This causes:
- Redundant nodes in the graph
- Lost relationships (source/target names don't match)

**Solution:**
1. **Pass 1** — Extract entities from all chunks (entity discovery)
2. **Deduplication** — Use LLM to merge duplicates into a canonical entity list
3. **Pass 2** — Re-extract relationships, constraining the model to use only canonical entity names

In [26]:
ENTITY_EXTRACTION_PROMPT = """Extract all named entities from the following text.
Return a JSON object with:
- "entities": list of objects with keys: name (str), type (str), description (str)
  - name: the entity's proper name as mentioned in the text
  - type: a short uppercase label that best categorizes this entity (e.g., PERSON, ORGANIZATION, CONCEPT, LOCATION, METRIC, EVENT, PRODUCT, RISK, DATE, TECHNOLOGY, REGULATION, PROCESS, etc.)
  - description: one sentence describing the entity based on the text

Assign whatever type best fits each entity - do not limit yourself to a fixed set of types.
Only extract entities explicitly mentioned in the text.
Return ONLY valid JSON, no other text.

Text:
"""

def extract_entities_pass1(chunk_text):
    """Pass 1: Extract entities only (no relationships yet)."""
    response = bedrock_runtime.converse(
        modelId=EXTRACTION_MODEL,
        messages=[{
            "role": "user",
            "content": [{"text": ENTITY_EXTRACTION_PROMPT + chunk_text}]
        }],
        inferenceConfig={"maxTokens": 2048, "temperature": 0}
    )
    
    result_text = response['output']['message']['content'][0]['text']
    
    try:
        if "```json" in result_text:
            result_text = result_text.split("```json")[1].split("```")[0]
        elif "```" in result_text:
            result_text = result_text.split("```")[1].split("```")[0]
        return json.loads(result_text)
    except json.JSONDecodeError:
        return {"entities": []}

# Test with first chunk
test_result = extract_entities_pass1(chunks[0]['text'])
print(f"Entities found: {len(test_result.get('entities', []))}")
print("\nSample entities:")
for e in test_result.get('entities', [])[:5]:
    print(f"  - {e['name']} ({e['type']})")

Entities found: 6

Sample entities:
  - Octank Financial (ORGANIZATION)
  - 10K report (DOCUMENT)
  - Schedule of Assets (FINANCIAL_SCHEDULE)
  - Schedule of Liabilities (FINANCIAL_SCHEDULE)
  - Schedule of Stockholders' Equity (FINANCIAL_SCHEDULE)


In [27]:
# === PASS 1: Extract entities from all chunks ===
raw_entities = {}  # normalized_name -> entity info (keeps first seen description)
chunk_entities_map = {}  # chunk_id -> list of raw entity names (for MENTIONS later)

def normalize_name(name):
    """Basic normalization for initial grouping."""
    return name.strip().title()

def process_chunk_pass1(chunk):
    """Extract entities from a single chunk."""
    result = extract_entities_pass1(chunk['text'])
    return chunk['id'], result

print(f"Pass 1: Extracting entities from {len(chunks)} chunks...")

with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(process_chunk_pass1, chunk): chunk for chunk in chunks}
    
    completed = 0
    for future in concurrent.futures.as_completed(futures):
        completed += 1
        if completed % 25 == 0:
            print(f"  Completed {completed}/{len(chunks)}...")
        
        chunk_id, result = future.result()
        
        entity_names = []
        for entity in result.get('entities', []):
            name = normalize_name(entity['name'])
            entity_names.append(name)
            if name not in raw_entities:
                raw_entities[name] = {
                    'name': name,
                    'type': entity.get('type', 'UNKNOWN'),
                    'description': entity.get('description', '')
                }
        
        chunk_entities_map[chunk_id] = entity_names

print(f"\nPass 1 complete!")
print(f"Raw unique entities (after basic normalization): {len(raw_entities)}")
type_counts = Counter(e['type'] for e in raw_entities.values())
for t, c in type_counts.most_common():
    print(f"  {t}: {c}")

Pass 1: Extracting entities from 175 chunks...
  Completed 25/175...
  Completed 50/175...
  Completed 75/175...
  Completed 100/175...
  Completed 125/175...
  Completed 150/175...
  Completed 175/175...

Pass 1 complete!
Raw unique entities (after basic normalization): 531
  METRIC: 132
  PROCESS: 50
  ORGANIZATION: 42
  PRODUCT: 35
  RISK: 32
  PERSON: 24
  LOCATION: 21
  DATE: 20
  CONCEPT: 18
  REGULATION: 17
  SERVICE: 12
  ROLE: 10
  DOCUMENT: 9
  COMMITTEE: 7
  TECHNOLOGY: 7
  FINANCIAL_METRIC: 7
  ASSET_CLASS: 6
  LEGAL_PROCEEDING: 5
  VALUE: 5
  PROFESSION: 5
  POSITION: 5
  FINANCIAL_SCHEDULE: 4
  SECTOR: 4
  BORROWER_CATEGORY: 4
  CLIENT_TYPE: 4
  COMPENSATION_COMPONENT: 3
  COMPENSATION_TYPE: 3
  ASSET: 3
  ESTIMATE: 3
  EVENT: 3
  PRICING_MODEL: 3
  FINANCIAL_STATEMENT: 3
  CLASSIFICATION: 2
  FINANCIAL_DOCUMENT: 2
  BUILDING: 2
  BUSINESS_DIVISION: 2
  FOCUS_AREA: 2
  POLICY: 1
  CURRENCY: 1
  EXHIBIT: 1
  LEGAL_DOCUMENT: 1
  GROUP: 1
  INDUSTRY: 1
  ACCOUNTING_CONCEPT: 

### 3b. Deduplicate Entities with LLM

Group entities by type and ask the LLM to merge duplicates into canonical names.
This resolves cases like "Credit Risk" / "Credit risk" that survive basic `.title()` normalization,
as well as semantic duplicates like "Octank" vs "Octank Financial".

In [28]:
DEDUP_PROMPT = """You are given a list of entity names of type "{entity_type}" extracted from a document.
Many entries refer to the same real-world entity but with slightly different names.

Your task: group duplicates together and choose ONE canonical name for each group.
- Keep the most complete and specific form (e.g., "Securities and Exchange Commission" over "SEC")
- Merge abbreviations, casing variants, and obvious synonyms
- Do NOT merge entities that are genuinely different

Return a JSON object:
{{
  "groups": [
    {{"canonical": "Chosen Name", "aliases": ["variant1", "variant2", ...]}}
  ]
}}

Only group entities that are truly duplicates. Entities with no duplicates should NOT appear in the output.
Return ONLY valid JSON.

Entity names:
{entity_list}
"""

def deduplicate_entity_group(entity_type, entity_names):
    """Use LLM to find and merge duplicate entities within a type."""
    if len(entity_names) <= 3:
        return {}
    
    batch_size = 150
    all_mappings = {}
    
    for i in range(0, len(entity_names), batch_size):
        batch = entity_names[i:i + batch_size]
        prompt = DEDUP_PROMPT.format(
            entity_type=entity_type,
            entity_list=json.dumps(batch, indent=1)
        )
        
        response = bedrock_runtime.converse(
            modelId=EXTRACTION_MODEL,
            messages=[{"role": "user", "content": [{"text": prompt}]}],
            inferenceConfig={"maxTokens": 4096, "temperature": 0}
        )
        
        result_text = response['output']['message']['content'][0]['text']
        
        try:
            if "```json" in result_text:
                result_text = result_text.split("```json")[1].split("```")[0]
            elif "```" in result_text:
                result_text = result_text.split("```")[1].split("```")[0]
            result = json.loads(result_text)
            
            for group in result.get('groups', []):
                canonical = normalize_name(group['canonical'])
                for alias in group.get('aliases', []):
                    all_mappings[normalize_name(alias)] = canonical
        except json.JSONDecodeError:
            continue
    
    return all_mappings

# Run deduplication per entity type
print("Deduplicating entities with LLM...")
entity_name_mapping = {}  # old_name -> canonical_name

entities_by_type = {}
for name, info in raw_entities.items():
    t = info['type']
    if t not in entities_by_type:
        entities_by_type[t] = []
    entities_by_type[t].append(name)

for entity_type, names in entities_by_type.items():
    print(f"  {entity_type}: {len(names)} entities...", end="")
    mappings = deduplicate_entity_group(entity_type, names)
    entity_name_mapping.update(mappings)
    print(f" merged {len(mappings)} duplicates")

# Build canonical entity list
canonical_entities = {}
for name, info in raw_entities.items():
    canonical_name = entity_name_mapping.get(name, name)
    if canonical_name not in canonical_entities:
        canonical_entities[canonical_name] = {
            'name': canonical_name,
            'type': info['type'],
            'description': info['description']
        }

# Update chunk_entities_map to use canonical names
for chunk_id in chunk_entities_map:
    chunk_entities_map[chunk_id] = list(set(
        entity_name_mapping.get(name, name)
        for name in chunk_entities_map[chunk_id]
    ))

print(f"\nDeduplication complete!")
print(f"  Before: {len(raw_entities)} entities")
print(f"  After:  {len(canonical_entities)} canonical entities")
print(f"  Merged: {len(entity_name_mapping)} aliases")
print(f"\nSample merges:")
for alias, canonical in list(entity_name_mapping.items())[:10]:
    print(f"  '{alias}' -> '{canonical}'")

Deduplicating entities with LLM...
  ORGANIZATION: 42 entities...

 merged 7 duplicates
  DOCUMENT: 9 entities... merged 4 duplicates
  FINANCIAL_SCHEDULE: 4 entities... merged 0 duplicates
  PERSON: 24 entities... merged 4 duplicates
  DATE: 20 entities... merged 2 duplicates
  METRIC: 132 entities... merged 8 duplicates
  PROCESS: 50 entities... merged 8 duplicates
  CONCEPT: 18 entities... merged 2 duplicates
  COMMITTEE: 7 entities... merged 4 duplicates
  COMPENSATION_COMPONENT: 3 entities... merged 0 duplicates
  REGULATION: 17 entities... merged 4 duplicates
  COMPENSATION_TYPE: 3 entities... merged 0 duplicates
  POLICY: 1 entities... merged 0 duplicates
  CURRENCY: 1 entities... merged 0 duplicates
  EXHIBIT: 1 entities... merged 0 duplicates
  ROLE: 10 entities... merged 0 duplicates
  LEGAL_DOCUMENT: 1 entities... merged 0 duplicates
  RISK: 32 entities... merged 12 duplicates
  PRODUCT: 35 entities... merged 5 duplicates
  CLASSIFICATION: 2 entities... merged 0 duplicates
  ASSET: 3 entities... merged 0 duplicates
  FINANCIAL_DOCUMENT: 2 e

### 3c. Extract Relationships (Pass 2 - Constrained)

Now we re-process each chunk, but this time the model receives the **canonical entity list** 
and must pick source/target from that list. This eliminates mismatched references.

In [29]:
RELATIONSHIP_EXTRACTION_PROMPT = """Extract relationships between entities in the following text.

IMPORTANT: You MUST use entity names EXACTLY as they appear in the canonical entity list below.
Do NOT invent new entity names. Only create relationships between entities from this list.

Canonical entity list:
{entity_list}

Return a JSON object with:
- "relationships": list of objects with keys:
  - source (str): entity name from the list above
  - target (str): entity name from the list above
  - type (str): a short uppercase label describing the relationship (e.g., WORKS_AT, MANAGES, CONTAINS, CAUSES, PRODUCES, LOCATED_IN, PART_OF, IMPACTS, REGULATES, DEPENDS_ON, etc.)
  - description (str): brief description of the relationship

Assign whatever relationship type best describes the connection - do not limit yourself to a fixed set.
Only extract relationships explicitly stated in the text. Source and target MUST be from the canonical list.
Return ONLY valid JSON, no other text.

Text:
{chunk_text}
"""

def extract_relationships_pass2(chunk_text, relevant_entities):
    """Pass 2: Extract relationships constrained to canonical entity names."""
    entity_list_str = json.dumps(relevant_entities)
    
    prompt = RELATIONSHIP_EXTRACTION_PROMPT.format(
        entity_list=entity_list_str,
        chunk_text=chunk_text
    )
    
    response = bedrock_runtime.converse(
        modelId=EXTRACTION_MODEL,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 2048, "temperature": 0}
    )
    
    result_text = response['output']['message']['content'][0]['text']
    
    try:
        if "```json" in result_text:
            result_text = result_text.split("```json")[1].split("```")[0]
        elif "```" in result_text:
            result_text = result_text.split("```")[1].split("```")[0]
        return json.loads(result_text)
    except json.JSONDecodeError:
        return {"relationships": []}

# Build lookup for which canonical entities appear in each chunk
canonical_entity_set = set(canonical_entities.keys())

def process_chunk_pass2(chunk):
    """Extract relationships for a chunk using its known entities."""
    chunk_id = chunk['id']
    relevant = chunk_entities_map.get(chunk_id, [])
    if len(relevant) < 2:
        return chunk_id, {"relationships": []}
    
    result = extract_relationships_pass2(chunk['text'], relevant)
    return chunk_id, result

# Run Pass 2
print(f"Pass 2: Extracting relationships from {len(chunks)} chunks (constrained to canonical entities)...")

all_relationships = []

with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(process_chunk_pass2, chunk): chunk for chunk in chunks}
    
    completed = 0
    valid = 0
    invalid = 0
    for future in concurrent.futures.as_completed(futures):
        completed += 1
        if completed % 25 == 0:
            print(f"  Completed {completed}/{len(chunks)} (valid rels: {valid}, rejected: {invalid})...")
        
        chunk_id, result = future.result()
        
        for rel in result.get('relationships', []):
            source = rel.get('source', '').strip()
            target = rel.get('target', '').strip()
            
            # Validate that both source and target are canonical entities
            if source in canonical_entity_set and target in canonical_entity_set and source != target:
                rel['chunk_id'] = chunk_id
                all_relationships.append(rel)
                valid += 1
            else:
                invalid += 1

print(f"\nPass 2 complete!")
print(f"  Valid relationships: {valid}")
print(f"  Rejected (non-canonical names): {invalid}")
print(f"  Rejection rate: {invalid/(valid+invalid)*100:.1f}%" if (valid+invalid) > 0 else "  No relationships extracted")
print(f"\nRelationship types:")
rel_type_counts = Counter(r['type'] for r in all_relationships)
for t, c in rel_type_counts.most_common():
    print(f"  {t}: {c}")

Pass 2: Extracting relationships from 175 chunks (constrained to canonical entities)...


  Completed 25/175 (valid rels: 210, rejected: 1)...
  Completed 50/175 (valid rels: 452, rejected: 9)...
  Completed 75/175 (valid rels: 665, rejected: 9)...
  Completed 100/175 (valid rels: 848, rejected: 9)...
  Completed 125/175 (valid rels: 1063, rejected: 9)...
  Completed 150/175 (valid rels: 1305, rejected: 13)...
  Completed 175/175 (valid rels: 1522, rejected: 14)...

Pass 2 complete!
  Valid relationships: 1531
  Rejected (non-canonical names): 14
  Rejection rate: 0.9%

Relationship types:
  PART_OF: 94
  WORKS_AT: 85
  IMPACTS: 77
  MEASURED_IN: 67
  MANAGES: 42
  PROVIDES: 41
  CONTAINS: 39
  MEASURED_AT: 30
  EXPOSED_TO: 27
  LOCATED_IN: 26
  SUPPORTS: 23
  PRODUCES: 22
  USES: 21
  REPORTS: 21
  OFFERS: 20
  EMPLOYS: 19
  HOLDS: 17
  MEMBER_OF: 17
  ASSOCIATED_WITH: 15
  REPORTED_FOR_PERIOD: 15
  EMBODIES: 15
  FOCUSES_ON: 14
  COMPONENT_OF: 13
  OWNS: 13
  RECEIVES: 13
  REGULATES: 12
  SIGNS: 12
  BELONGS_TO: 11
  CONTRIBUTES_TO: 11
  COVERS_PERIOD: 10
  MEASURED_BY: 

## 4. Create Neptune Analytics Graph

In [30]:
timestamp_str = time.strftime("%Y%m%d%H%M%S", time.localtime())[-7:]
graph_name = f"manual-graph-rag-{timestamp_str}"

print(f"Creating Neptune Analytics graph: {graph_name}")
print("This takes 3-5 minutes...")

response = neptune_client.create_graph(
    graphName=graph_name,
    tags={'usecase': 'manual-graphRAG'},
    publicConnectivity=True,
    vectorSearchConfiguration={'dimension': EMBEDDING_DIMENSION},
    replicaCount=0,
    deletionProtection=False,
    provisionedMemory=16
)

graph_id = response['id']
print(f"Graph ID: {graph_id}")

# Wait for graph to be available
while True:
    status = neptune_client.get_graph(graphIdentifier=graph_id)['status']
    if status == 'AVAILABLE':
        print(f"Graph is AVAILABLE!")
        break
    print(f"  Status: {status}...")
    time.sleep(30)

Creating Neptune Analytics graph: manual-graph-rag-7153729
This takes 3-5 minutes...
Graph ID: g-6dpxfkxlo6
  Status: CREATING...
  Status: CREATING...
  Status: CREATING...
  Status: CREATING...
  Status: CREATING...
  Status: CREATING...
  Status: CREATING...
Graph is AVAILABLE!


## 5. Generate Embeddings and Populate Graph

We'll insert:
- **Chunk nodes** with text content and vector embeddings
- **Entity nodes** with type and description
- **Relationships** between entities
- **MENTIONS** edges linking chunks to entities

In [31]:
def get_embedding(text):
    """Generate embedding using Titan Embeddings V2."""
    response = bedrock_runtime.invoke_model(
        modelId=EMBEDDING_MODEL,
        body=json.dumps({
            "inputText": text[:8000],  # Titan V2 limit
            "dimensions": EMBEDDING_DIMENSION,
            "normalize": True
        })
    )
    return json.loads(response['body'].read())['embedding']

# Test
test_emb = get_embedding("test")
print(f"Embedding dimension: {len(test_emb)}")

Embedding dimension: 1024


In [32]:
def run_query(query, parameters=None):
    """Execute an openCypher query against Neptune Analytics."""
    kwargs = {
        'graphIdentifier': graph_id,
        'queryString': query,
        'language': 'OPEN_CYPHER'
    }
    if parameters:
        kwargs['parameters'] = parameters
    
    response = neptune_client.execute_query(**kwargs)
    return response['payload'].read().decode('utf-8')

In [34]:
# Insert chunk nodes with embeddings (parallel)
def insert_chunk(chunk):
    """Insert a single chunk node + embedding."""
    embedding = get_embedding(chunk['text'])
    embedding_str = "[" + ",".join(str(x) for x in embedding) + "]"
    
    run_query(
        "MERGE (c:Chunk {id: $id}) SET c.text = $text, c.start_pos = $start_pos, c.end_pos = $end_pos",
        parameters={
            'id': chunk['id'],
            'text': chunk['text'],
            'start_pos': chunk['start'],
            'end_pos': chunk['end']
        }
    )
    
    run_query(f"""
        MATCH (c:Chunk {{id: '{chunk["id"]}'}})
        CALL neptune.algo.vectors.upsert(c, {embedding_str})
        YIELD success
        RETURN success
    """)
    return chunk['id']

print(f"Inserting {len(chunks)} chunk nodes with embeddings (10 workers)...")

with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(insert_chunk, chunk): chunk for chunk in chunks}
    completed = 0
    for future in concurrent.futures.as_completed(futures):
        completed += 1
        if completed % 10 == 0:
            print(f"  Completed {completed}/{len(chunks)}...")
        future.result()

print(f"Done inserting {len(chunks)} chunks!")

Inserting 175 chunk nodes with embeddings (10 workers)...
  Completed 10/175...
  Completed 20/175...
  Completed 30/175...
  Completed 40/175...
  Completed 50/175...
  Completed 60/175...
  Completed 70/175...
  Completed 80/175...
  Completed 90/175...
  Completed 100/175...
  Completed 110/175...
  Completed 120/175...
  Completed 130/175...
  Completed 140/175...
  Completed 150/175...
  Completed 160/175...
  Completed 170/175...
Done inserting 175 chunks!


In [35]:
# Insert entity nodes (canonical, deduplicated)
print(f"Inserting {len(canonical_entities)} canonical entity nodes...")

for i, (name, entity) in enumerate(canonical_entities.items()):
    if i % 50 == 0:
        print(f"  Entity {i+1}/{len(canonical_entities)}...")
    
    query = """
    MERGE (e:Entity {name: $name})
    SET e.type = $type,
        e.description = $description
    """
    
    run_query(query, parameters={
        'name': entity['name'],
        'type': entity.get('type', 'UNKNOWN'),
        'description': entity.get('description', '')
    })

print("Done inserting entities!")

Inserting 463 canonical entity nodes...
  Entity 1/463...


  Entity 51/463...
  Entity 101/463...
  Entity 151/463...
  Entity 201/463...
  Entity 251/463...
  Entity 301/463...
  Entity 351/463...
  Entity 401/463...
  Entity 451/463...
Done inserting entities!


In [36]:
# Insert relationships (all validated against canonical entity set in Pass 2)
print(f"Inserting {len(all_relationships)} validated relationships...")

inserted = 0
for i, rel in enumerate(all_relationships):
    if i % 50 == 0 and i > 0:
        print(f"  Relationship {i}/{len(all_relationships)} (inserted: {inserted})...")
    
    query = """
    MATCH (s:Entity {name: $source})
    MATCH (t:Entity {name: $target})
    MERGE (s)-[r:RELATES_TO {type: $rel_type}]->(t)
    SET r.description = $description
    """
    
    run_query(query, parameters={
        'source': rel['source'],
        'target': rel['target'],
        'rel_type': rel.get('type', 'RELATED'),
        'description': rel.get('description', '')
    })
    inserted += 1

print(f"\nInserted {inserted} relationships (100% match rate - all pre-validated)")

Inserting 1531 validated relationships...
  Relationship 50/1531 (inserted: 50)...
  Relationship 100/1531 (inserted: 100)...
  Relationship 150/1531 (inserted: 150)...
  Relationship 200/1531 (inserted: 200)...
  Relationship 250/1531 (inserted: 250)...
  Relationship 300/1531 (inserted: 300)...
  Relationship 350/1531 (inserted: 350)...
  Relationship 400/1531 (inserted: 400)...
  Relationship 450/1531 (inserted: 450)...
  Relationship 500/1531 (inserted: 500)...
  Relationship 550/1531 (inserted: 550)...
  Relationship 600/1531 (inserted: 600)...
  Relationship 650/1531 (inserted: 650)...
  Relationship 700/1531 (inserted: 700)...
  Relationship 750/1531 (inserted: 750)...
  Relationship 800/1531 (inserted: 800)...
  Relationship 850/1531 (inserted: 850)...
  Relationship 900/1531 (inserted: 900)...
  Relationship 950/1531 (inserted: 950)...
  Relationship 1000/1531 (inserted: 1000)...
  Relationship 1050/1531 (inserted: 1050)...
  Relationship 1100/1531 (inserted: 1100)...
  Relati

In [37]:
# Link chunks to entities with MENTIONS edges
print("Creating MENTIONS edges (chunk -> entity)...")

mentions_count = 0
for chunk_id, entity_names in chunk_entities_map.items():
    for entity_name in entity_names:
        query = """
        MATCH (c:Chunk {id: $chunk_id})
        MATCH (e:Entity {name: $entity_name})
        MERGE (c)-[:MENTIONS]->(e)
        """
        try:
            run_query(query, parameters={
                'chunk_id': chunk_id,
                'entity_name': entity_name
            })
            mentions_count += 1
        except Exception:
            pass

print(f"Created {mentions_count} MENTIONS edges")

Creating MENTIONS edges (chunk -> entity)...


Created 1637 MENTIONS edges


In [38]:
# Verify graph contents
result = run_query("MATCH (n) RETURN labels(n) AS label, count(n) AS count")
print("Node counts:")
print(result)

print("\nRelationship counts:")
result = run_query("MATCH ()-[r]->() RETURN type(r) AS type, count(r) AS count")
print(result)

Node counts:
{
  "results": [{
      "label": ["Chunk"],
      "count": 175
    }, {
      "label": ["Entity"],
      "count": 463
    }]
}

Relationship counts:
{
  "results": [{
      "type": "RELATES_TO",
      "count": 1199
    }, {
      "type": "MENTIONS",
      "count": 1637
    }]
}


## 6. Query: Vector Similarity + Graph Traversal

The hybrid query strategy:
1. **Vector search** — find the most relevant chunks by embedding similarity
2. **Graph traversal** — follow MENTIONS edges to find related entities and their connections
3. **Context enrichment** — gather connected chunks for broader context

In [41]:
def hybrid_search(query_text, top_k=5, max_entities=10):
    """Perform hybrid vector + graph search with optimized entity retrieval."""
    
    # Step 1: Vector similarity search on chunks
    query_embedding = get_embedding(query_text)
    embedding_str = "[" + ",".join(str(x) for x in query_embedding) + "]"
    
    vector_query = f"""
    CALL neptune.algo.vectors.topKByEmbedding(
        {embedding_str},
        {{topK: {top_k}}}
    )
    YIELD node, score
    WHERE node:Chunk
    RETURN node.id AS chunk_id, node.text AS text, score
    ORDER BY score DESC
    """
    
    vector_results = json.loads(run_query(vector_query))
    
    # Step 2: Get entities mentioned in top chunks, ranked by connectivity
    chunk_ids = [r['chunk_id'] for r in vector_results.get('results', [])]
    
    if chunk_ids:
        # Only get entities that have connections (filter isolated ones)
        # Rank by number of connections to prioritize high-value entities
        entity_query = f"""
        MATCH (c:Chunk)-[:MENTIONS]->(e:Entity)
        WHERE c.id IN $chunk_ids
        OPTIONAL MATCH (e)-[r:RELATES_TO]-(other:Entity)
        WITH e, collect(DISTINCT {{related: other.name, relation: r.type}}) AS connections
        WHERE size(connections) > 0
        RETURN e.name AS entity, e.type AS type,
               substring(e.description, 0, 100) AS description,
               connections[..5] AS connections,
               size(connections) AS relevance
        ORDER BY relevance DESC
        LIMIT {max_entities}
        """
        
        entity_results = json.loads(run_query(entity_query, parameters={
            'chunk_ids': chunk_ids
        }))
    else:
        entity_results = {'results': []}
    
    return {
        'chunks': vector_results.get('results', []),
        'entities': entity_results.get('results', [])
    }

# Test search
test_results = hybrid_search("What are the main risks for Octank Financial?")
print(f"Found {len(test_results['chunks'])} relevant chunks")
print(f"Found {len(test_results['entities'])} related entities (capped at 10)")
print("\nTop entities:")
for e in test_results['entities'][:5]:
    print(f"  - {e['entity']} ({e['type']}) [{e['relevance']} connections]")

Found 5 relevant chunks
Found 10 related entities (capped at 10)

Top entities:
  - Octank Financial (ORGANIZATION) [431 connections]
  - Personb (PERSON) [29 connections]
  - Persona (PERSON) [28 connections]
  - Personc (PERSON) [23 connections]
  - Credit Risk (RISK) [12 connections]


## 7. Generate Answers with Graph-Enriched Context

In [42]:
def graph_rag_query(question, top_k=5, max_entities=10):
    """Full Graph RAG: search + generate with optimized context."""
    
    # Retrieve
    results = hybrid_search(question, top_k=top_k, max_entities=max_entities)
    
    # Build chunk context
    chunk_context = "\n\n".join([
        f"[Chunk {i+1} | Score: {r['score']:.3f}]\n{r['text']}"
        for i, r in enumerate(results['chunks'])
    ])
    
    # Build compact entity context (truncated descriptions, limited connections)
    entity_lines = []
    for e in results['entities']:
        connections = [c['related'] for c in e.get('connections', []) if c.get('related')]
        conn_str = f" -> {', '.join(connections)}" if connections else ""
        desc = e.get('description', '')
        entity_lines.append(f"- {e['entity']} ({e['type']}): {desc}{conn_str}")
    entity_context = "\n".join(entity_lines)
    
    # Generate
    prompt = f"""Answer the question using ONLY the provided context. Include specific details and numbers.

## Retrieved Document Chunks
{chunk_context}

## Knowledge Graph (top entities and connections)
{entity_context}

## Question
{question}"""
    
    response = bedrock_runtime.converse(
        modelId=GENERATION_MODEL,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 1024, "temperature": 0}
    )
    
    answer = response['output']['message']['content'][0]['text']
    usage = response['usage']
    
    return {
        'answer': answer,
        'chunks_used': len(results['chunks']),
        'entities_found': len(results['entities']),
        'input_tokens': usage['inputTokens'],
        'output_tokens': usage['outputTokens'],
        'total_tokens': usage['inputTokens'] + usage['outputTokens']
    }

In [43]:
# Test Query 1
question = "Provide a summary of consolidated statements of cash flows of Octank Financial for the fiscal years ended December 31, 2019."

result = graph_rag_query(question)
print(f"Chunks used: {result['chunks_used']} | Entities found: {result['entities_found']} | Tokens: {result['input_tokens']} in + {result['output_tokens']} out = {result['total_tokens']} total\n")
display(Markdown(result['answer']))

Chunks used: 5 | Entities found: 10 | Tokens: 2606 in + 384 out = 2990 total



# Summary of Octank Financial's Consolidated Statements of Cash Flows
## Fiscal Year Ended December 31, 2019

### Cash Flows from Operating Activities
| Item | Amount |
|------|--------|
| Net Income | $700 million |
| Depreciation and amortization | $190 million |
| Stock-based compensation | $80 million |
| Deferred income tax benefit | ($20 million) |
| Accounts receivable | ($60 million) |
| Inventory | ($110 million) |
| Prepaid expenses and other current assets | $30 million |
| Accounts payable | $60 million |
| Accrued liabilities and other | $40 million |
| **Net Cash Provided by Operating Activities** | **$710 million** |

### Cash Flows from Investing Activities
| Item | Amount |
|------|--------|
| Purchases of property, plant, and equipment | ($200 million) |
| Proceeds from sales of property, plant, and equipment | $40 million |
| Purchases of marketable securities | ($60 million) |
| Maturing marketable securities | $20 million |
| **Net Cash Used in Investing Activities** | **($240 million)** |

### Cash Flows from Financing Activities
- Net cash **provided** by financing activities: **$350 million**
- Primarily driven by proceeds from issuance of common stock and long-term debt

### Overall Cash Position
- **Net increase in cash and cash equivalents:** $120 million
- **Cash and cash equivalents at end of 2019:** $210 million

In [44]:
# Test Query 2
question = "What are the main risks faced by Octank Financial and how are they connected?"

result = graph_rag_query(question, top_k=8)
print(f"Chunks used: {result['chunks_used']} | Entities found: {result['entities_found']} | Tokens: {result['input_tokens']} in + {result['output_tokens']} out = {result['total_tokens']} total\n")
display(Markdown(result['answer']))

Chunks used: 8 | Entities found: 10 | Tokens: 3150 in + 648 out = 3798 total



# Main Risks Faced by Octank Financial and Their Interconnections

## Core Risk Categories

### 1. Market Risk
Octank Financial faces volatility from **changes in interest rates, inflation, economic downturns, political instability, and natural disasters**. A notable example is the trade tensions between the US and China, which significantly impacted their portfolio's performance. This risk directly connects to **Equity Price Risk** and **Foreign Exchange Risk**.

### 2. Credit Risk
The company is exposed to potential defaults by debt security issuers. A concrete example cited is **XYZ Inc.**, a high-yield bond issuer that defaulted on its obligations, creating a **ripple effect in the high-yield bond market** — directly linking Credit Risk to Liquidity Risk.

### 3. Liquidity Risk
During the **COVID-19 pandemic**, the market for **commercial mortgage-backed securities (CMBS)** became illiquid, making fair-value sales challenging. This demonstrates how external events can trigger liquidity constraints.

### 4. Operational & Cybersecurity Risk
Past **cyber-attacks resulted in loss of sensitive data and financial losses**. As a processor of sensitive customer data, Octank remains a potential target, with risks of **reputational damage and legal liability**.

### 5. Regulatory & Compliance Risk
Non-compliance could result in **additional costs, activity restrictions, or penalties**. The Compliance Officer works to address this, but even a **single regulatory failure** could materially affect financial condition.

### 6. Strategic Risk
The **expansion into the European market** exposed Octank to new competitors and regulatory challenges, directly affecting financial results.

### 7. Legal Risk
Octank is currently involved in a **wrongful termination lawsuit** from a former employee, illustrating active legal exposure.

### 8. ESG Risk
An **investment in a coal-fired power plant** has created reputational and financial risks due to climate change concerns, while an **investment in a Brazilian manufacturing company** has introduced **currency volatility risk** via the Brazilian real.

---

## Risk Interconnections

| Risk | Connected To | How |
|------|-------------|-----|
| Market Risk | Credit & Liquidity Risk | Market downturns increase defaults and reduce asset liquidity |
| Credit Risk | Liquidity Risk | Defaults (e.g., XYZ Inc.) create illiquidity in bond markets |
| Operational Risk | Legal & Reputation Risk | Cyber-attacks lead to lawsuits and reputational damage |
| ESG Risk | Reputation & Strategic Risk | Coal plant investment damages reputation and strategic positioning |
| Strategic Risk | Regulatory Risk | European expansion introduced new regulatory challenges |

These risks are **deeply interconnected**, where a single event can cascade across multiple risk categories, potentially creating a **material adverse effect** on Octank Financial's overall financial condition and reputation.

In [45]:
# Test Query 3 - Entity-centric query
question = "Who are the key personnel at Octank Financial and what are their roles?"

result = graph_rag_query(question, max_entities=20)
print(f"Chunks used: {result['chunks_used']} | Entities found: {result['entities_found']} | Tokens: {result['input_tokens']} in + {result['output_tokens']} out = {result['total_tokens']} total\n")
display(Markdown(result['answer']))

Chunks used: 5 | Entities found: 20 | Tokens: 2641 in + 436 out = 3077 total



## Key Personnel at Octank Financial

Based on the provided context, here are the key personnel at Octank Financial and their roles:

### Board Leadership
1. **John Doe** - *Chairman of the Board*: Has been with Octank Financial for over **20 years** and has served as Chairman for the past **10 years**
2. **Jane Smith** - *Vice Chairman*: Has been with the company for **15 years** and has served as Vice Chairman for the past **5 years**

### Executive Leadership (C-Suite)
3. **Michael Johnson** - *Chief Executive Officer (CEO)*: Has been with Octank Financial for **10 years** and has served as CEO for the past **5 years**, with a strong background in driving financial performance
4. **Sarah Lee** - *Chief Financial Officer (CFO)*: Has been with the company for **8 years** and has served as CFO for the past **3 years**, specializing in financial planning and budgeting
5. **David Kim** - *Chief Operating Officer (COO)*: Has been with the company for **7 years** and has served as COO for the past **2 years**, focused on operational efficiency
6. **Emily Brown** - *Chief Marketing Officer (CMO)*: Has been with the company for **6 years** and has served as CMO for the past **2 years**
7. **Daniel Park** - *Chief Information Officer (CIO)*: Has been with the company for **5 years**, driving digital transformation
8. **Jessica Kim** - *Chief Legal Officer (CLO)*: Has been with the company for **4 years**, overseeing legal and regulatory compliance
9. **Robert Lee** - *Chief Human Resources Officer (CHRO)*

### Founders
The company was **founded in 2010** by **PersonA**, **PersonB**, and **PersonC**, who remain significant figures within the organization.

## Explore the Graph

You can run custom openCypher queries to explore the knowledge graph.

In [46]:
# Show all entity types and counts
result = run_query("""
    MATCH (e:Entity)
    RETURN e.type AS type, count(e) AS count, collect(e.name)[..5] AS examples
    ORDER BY count DESC
""")
print("Entity types in graph:")
print(result)

Entity types in graph:
{
  "results": [{
      "type": "METRIC",
      "count": 124,
      "examples": ["12%", "$1.2 Billion", "$50 Million", "$250 Million", "1.6%"]
    }, {
      "type": "PROCESS",
      "count": 42,
      "examples": ["Peer Group Comparison", "Say-On-Pay Vote", "Scenario Analysis", "Credit Rating Migrations", "Training Program"]
    }, {
      "type": "ORGANIZATION",
      "count": 35,
      "examples": ["Lmn Inc.", "Company B", "Investment Group Y", "Disclosure Committee", "Legal Team"]
    }, {
      "type": "PRODUCT",
      "count": 31,
      "examples": ["Held-To-Maturity", "Commercial Paper", "Cash And Cash Equivalents", "Accounts Receivable", "Mutual Funds"]
    }, {
      "type": "RISK",
      "count": 21,
      "examples": ["Strategic Risk", "Legal Risk", "Foreign Exchange Risk", "Dependence On Key Personnel", "Credit Risk"]
    }, {
      "type": "PERSON",
      "count": 20,
      "examples": ["Persond", "Daniel Park", "Robert Lee", "Thomas Johnson", "Perso

In [47]:
# Find most connected entities
result = run_query("""
    MATCH (e:Entity)-[r]-()
    RETURN e.name AS entity, e.type AS type, count(r) AS connections
    ORDER BY connections DESC
    LIMIT 10
""")
print("Most connected entities:")
print(result)

Most connected entities:
{
  "results": [{
      "entity": "Octank Financial",
      "type": "ORGANIZATION",
      "connections": 576
    }, {
      "entity": "Octank Labs",
      "type": "ORGANIZATION",
      "connections": 102
    }, {
      "entity": "Persona",
      "type": "PERSON",
      "connections": 72
    }, {
      "entity": "Personb",
      "type": "PERSON",
      "connections": 71
    }, {
      "entity": "Consolidated Financial Statements",
      "type": "DOCUMENT",
      "connections": 70
    }, {
      "entity": "December 31, 2021",
      "type": "DATE",
      "connections": 68
    }, {
      "entity": "Personc",
      "type": "PERSON",
      "connections": 61
    }, {
      "entity": "2021",
      "type": "DATE",
      "connections": 53
    }, {
      "entity": "10K Report",
      "type": "DOCUMENT",
      "connections": 50
    }, {
      "entity": "2020",
      "type": "DATE",
      "connections": 44
    }]
}


In [48]:
# Find paths between two entities
result = run_query("""
    MATCH path = (a:Entity {name: 'Octank Financial'})-[*1..3]-(b:Entity)
    WHERE b.type = 'RISK'
    RETURN b.name AS risk, length(path) AS distance, b.description AS description
    ORDER BY distance
    LIMIT 10
""")
print("Risks connected to Octank Financial:")
print(result)

Risks connected to Octank Financial:
{
  "results": [{
      "risk": "Cybersecurity Risk",
      "distance": 1,
      "description": "Potential threats to Octank Financial from cyber attacks that could result in data loss, theft, reputational damage, and legal liability."
    }, {
      "risk": "Dependence On Key Suppliers",
      "distance": 1,
      "description": "A risk to Octank Financial's operations and financial performance arising from reliance on critical suppliers that could fail or experience disruptions."
    }, {
      "risk": "Credit Risk",
      "distance": 1,
      "description": "The risk that the issuer of a security will default on its obligations."
    }, {
      "risk": "Equity Price Risk",
      "distance": 1,
      "description": "A type of market risk related to changes in equity prices affecting individual equity securities and equity index instruments."
    }, {
      "risk": "Insider Trading",
      "distance": 1,
      "description": "Allegations against se

## 8. Cleanup

Delete the Neptune Analytics graph to avoid charges.

In [ ]:
# Uncomment to delete the graph
# print(f"Deleting graph {graph_id}...")
# neptune_client.delete_graph(graphIdentifier=graph_id, skipSnapshot=True)
# print("Graph deletion initiated.")